# Hybrid Eval Results Demo

This notebook visualizes downloaded `olmo-eval` results from `local/results/.../all_results.json`.

Edit the config cell to change the result file, regex filters, model-class labels, or plotted evals.

In [ ]:
from __future__ import annotations

from collections import defaultdict
import json
import math
import re
import sys
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "aesthetics.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIGURES_DIR = PROJECT_ROOT / "figures"

import aesthetics as aes  # noqa: E402

# Checkpoint schedules from ../olmo-eval-internal/local/run_evals.py.
STEPS_A = [0, 7320, 8134, 15454, 16268, 30910, 32537, 61819, 65073, 123639, 130000, 130147]
STEPS_B = [0, 7321, 8135, 15455, 16269, 30911, 32538, 61823, 65077, 123646, 130000, 130154]

pd.set_option("display.max_colwidth", 140)


## Config

In [ ]:
RESULTS_PATH = PROJECT_ROOT / "results" / "jacksonp-hybrid-small-downstream-evals" / "all_results.json"

# Keep only jobs/models matching this regex. Set to None to include every downloaded result.
# RUN_REGEX = r"olmo3-hybrid-gdn-deux_1b_step\d+"
RUN_REGEX = None

# Optional regex applied to eval/task names after flattening. Set to None to include all evals.
TASK_REGEX = None

# Compute per-example score statistics from raw predictions/*.jsonl files.
COMPUTE_RAW_SCORE_STATS = True

# Overlay uncertainty on the line plots. Use "sem" or "ci95_normal".
SHOW_SCORE_ERROR_BARS = True
SCORE_ERROR_BAR = "ci95_normal"

# Name each plotted series from the experiment/model text. First matching pattern wins.
MODEL_CLASS_PATTERNS = [
    (r"olmo3-hybrid-gdn-deux_1b", "hybrid 1B"),
    (r"olmo3-baseline-jacksonp_1b", "transformer 1B"),
    (r"transformer-small-cx100_275m", "transformer 275M"),
    (r"hybrid-small-cx100_275m", "hybrid 275M"),
    (r"pure-gdn-ladder-gate-neg-eig_1b", "GDN± 1B"),
    (r"pure-gdn-ladder-gate-pos-eig_1b", "GDN+ 1B"),
    (r"allenai_olmo-3-1025-7b_stage1", "transformer 7B"),
    (r"allenai_olmo-hybrid-7b_stage1", "hybrid 7B"),
]

# Prefer aggregate suite scores when present; set to None to plot every eval in the file.
PREFERRED_TASKS = [
    # "monoid:commutative:3shot",
    # "monoid:soluble:3shot",
    # "monoid:insoluble:3shot",
    # "grapheme_en",

    "formal_langs_copy",
    "formal_langs_reverse_copy",
    "formal_langs_sort",
    
    "formal_langs_char_shift_1",
    "formal_langs_char_shift_2",
    "formal_langs_char_shift_5",

    # "formal_langs_k_dyck",
    # "formal_langs_k_shuffle_dyck",

    "formal_langs_k_dyck:k1",
    "formal_langs_k_dyck:k2",
    "formal_langs_k_dyck:k3",
    "formal_langs_k_dyck:k4",
    "formal_langs_k_dyck:k5",
    "formal_langs_k_dyck:k6",

    "formal_langs_k_shuffle_dyck:k1",
    "formal_langs_k_shuffle_dyck:k2",
    "formal_langs_k_shuffle_dyck:k3",
    "formal_langs_k_shuffle_dyck:k4",
    "formal_langs_k_shuffle_dyck:k5",
    "formal_langs_k_shuffle_dyck:k6",

    # "formal_langs:v2",
    # "formal_langs:v4",
    # "formal_langs:v6",
 
    "formal_langs_var_unique:v2",
    "formal_langs_var_unique:v4",
    "formal_langs_var_unique:v6",
    
    "formal_langs_var_reassign_const:v2",
    "formal_langs_var_reassign_const:v4",
    "formal_langs_var_reassign_const:v6",
    
    "formal_langs_var_reassign_var:v2",
    "formal_langs_var_reassign_var:v4",
    "formal_langs_var_reassign_var:v6",
    
    "formal_langs_var_undefined:v2",
    "formal_langs_var_undefined:v4",
    "formal_langs_var_undefined:v6",

    "formal_langs_cube_unique:v2",
    "formal_langs_cube_unique:v4",
    "formal_langs_cube_unique:v6",
    
    "formal_langs_cube_reassign_const:v2",
    "formal_langs_cube_reassign_const:v4",
    "formal_langs_cube_reassign_const:v6",
    
    "formal_langs_cube_reassign_var:v2",
    "formal_langs_cube_reassign_var:v4",
    "formal_langs_cube_reassign_var:v6",
    
    "formal_langs_cube_undefined:v2",
    "formal_langs_cube_undefined:v4",
    "formal_langs_cube_undefined:v6",

    "formal_langs_reverse_copy",
    "formal_langs_sort",
    "formal_langs_char_shift_1",
    "formal_langs_char_shift_1",
    "formal_langs_char_shift_5",
]

# PREFERRED_TASKS=None

# Hide the earlier checkpoint in each pre/post learning-rate-decay pair from run_evals.py.
# Keeps step 0 and any unpaired final checkpoint.
FILTER_PRE_DECAY_CHECKPOINTS = True

# If multiple jobs report the same model family / step / eval, keep only the latest experiment_id.
KEEP_LATEST_RUN_PER_MODEL_TASK = True

# If multiple jobs report the same model class / step / eval, average them for cleaner lines.
AGGREGATE_DUPLICATES = True

# Cube-only assignment breakdown plot settings.
# Set CUBE_ASSIGNMENT_BINS = None to plot each exact assignment count separately.
CUBE_ASSIGNMENT_BINS = [(2, 17), (18, 34), (35, 50)]

# Persistent regression outputs.
REGRESSION_RESULTS_DIR = PROJECT_ROOT / "analysis_outputs" / "cube_regressions"

## Load And Flatten

In [ ]:
def extract_step(*values: str | None) -> int | None:
    for value in values:
        if not value:
            continue
        match = re.search(r"(?:^|[_/-])step(\d+)(?:\D|$)", value, flags=re.IGNORECASE)
        if match:
            return int(match.group(1))
    return None


def model_class_for(experiment_name: str, model: str) -> str:
    haystack = f"{experiment_name} {model}"
    for pattern, label in MODEL_CLASS_PATTERNS:
        if re.search(pattern, haystack, flags=re.IGNORECASE):
            return label

    # Fallback: strip to a stable checkpoint family when the path follows .../<family>/<size>/stepN.
    parts = [part for part in Path(model.rstrip("/")).parts if part]
    if len(parts) >= 2 and re.match(r"step\d+", parts[-1], flags=re.IGNORECASE):
        return "/".join(parts[-3:-1]) if len(parts) >= 3 else parts[-2]
    return experiment_name.split("_step", 1)[0].split("-step", 1)[0]


def metric_value(instance_metrics: dict, metric: str | None) -> float | None:
    value = instance_metrics
    for key in (metric or "").split(":"):
        if not isinstance(value, dict) or key not in value:
            return None
        value = value[key]
    if isinstance(value, int | float) and pd.notna(value) and math.isfinite(value):
        return value
    return None


def task_by_name(result: dict) -> dict[str, dict]:
    return {task["task"]: task for task in result.get("tasks", [])}


def prediction_file_for_task(result: dict, task_name: str, tasks: dict[str, dict]) -> Path | None:
    task = tasks.get(task_name)
    task_hash = task.get("task_hash") if task else None
    if not task_hash:
        return None

    predictions_root = RESULTS_PATH.parent / result["experiment_id"] / "main" / "predictions"
    matches = sorted(predictions_root.glob(f"*/*_{task_hash[-6:]}-predictions.jsonl"))
    return matches[0] if matches else None


def component_tasks_for_aggregate(task_name: str, tasks: dict[str, dict]) -> list[str]:
    if ":" not in task_name:
        return []
    base_task, variant = task_name.rsplit(":", 1)
    prefix = f"{base_task}_"
    suffix = f":{variant}"
    return sorted(
        name
        for name in tasks
        if name.startswith(prefix) and name.endswith(suffix) and name != task_name
    )


def raw_scores_for_task(result: dict, task_name: str, metric: str | None, tasks: dict[str, dict]) -> list[float]:
    path = prediction_file_for_task(result, task_name, tasks)
    if path is None:
        return []

    scores = []
    with path.open() as f:
        for line in f:
            value = metric_value(json.loads(line).get("instance_metrics", {}), metric)
            if value is not None:
                scores.append(float(value))
    return scores


def raw_score_stats(result: dict, task_name: str, metric: str | None, tasks: dict[str, dict]) -> dict:
    scores = raw_scores_for_task(result, task_name, metric, tasks)
    if not scores and metric == "primary_score:average":
        for component_task in component_tasks_for_aggregate(task_name, tasks):
            component_metric = result["summary"][component_task]["metric"]
            scores.extend(raw_scores_for_task(result, component_task, component_metric, tasks))

    if not scores:
        return {
            "score_std": None,
            "score_sem": None,
            "score_ci95_normal": None,
            "score_n": 0,
            "score_sum": None,
            "score_sum_sq": None,
        }

    series = pd.Series(scores, dtype="float64")
    score_n = int(series.count())
    score_std = series.std(ddof=1)
    score_sem = score_std / math.sqrt(score_n) if score_n > 1 else None

    return {
        "score_std": score_std,
        "score_sem": score_sem,
        "score_ci95_normal": 1.96 * score_sem if score_sem is not None else None,
        "score_n": score_n,
        "score_sum": float(series.sum()),
        "score_sum_sq": float((series**2).sum()),
    }


def primary_score_rows(result: dict) -> list[dict]:
    rows = []
    experiment_name = result.get("experiment_name", "")
    model = result.get("model", "")
    step = extract_step(experiment_name, model)
    model_class = model_class_for(experiment_name, model)
    tasks = task_by_name(result)

    for task_name, payload in result.get("summary", {}).items():
        row = {
            "experiment_id": result.get("experiment_id"),
            "experiment_name": experiment_name,
            "experiment_group": result.get("experiment_group"),
            "status": result.get("status"),
            "model": model,
            "model_class": model_class,
            "step": step,
            "task": task_name,
            "metric": payload.get("metric"),
            "score": payload.get("score"),
        }
        if COMPUTE_RAW_SCORE_STATS:
            row.update(raw_score_stats(result, task_name, payload.get("metric"), tasks))
        rows.append(row)
    return rows


raw_results = json.loads(RESULTS_PATH.read_text())
df = pd.DataFrame(row for result in raw_results for row in primary_score_rows(result))
df["score"] = pd.to_numeric(df["score"], errors="coerce")
df["step"] = pd.to_numeric(df["step"], errors="coerce")
if COMPUTE_RAW_SCORE_STATS:
    for column in ["score_std", "score_sem", "score_ci95_normal", "score_n", "score_sum", "score_sum_sq"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

print(f"Loaded {len(raw_results):,} experiments and {len(df):,} eval rows from {RESULTS_PATH}")
df.tail()

In [ ]:
df[df["model_class"] == "hybrid 275M"]

## Apply Filters

In [ ]:
plot_df = df.copy()

if RUN_REGEX:
    run_mask = (
        plot_df["experiment_name"].str.contains(RUN_REGEX, case=False, regex=True, na=False)
        | plot_df["model"].str.contains(RUN_REGEX, case=False, regex=True, na=False)
    )
    plot_df = plot_df[run_mask]

if TASK_REGEX:
    plot_df = plot_df[plot_df["task"].str.contains(TASK_REGEX, case=False, regex=True, na=False)]

if PREFERRED_TASKS:
    preferred = plot_df[plot_df["task"].isin(PREFERRED_TASKS)]
    if not preferred.empty:
        plot_df = preferred

plot_df = plot_df.dropna(subset=["step", "score"])


def pre_decay_checkpoints(steps: list[int]) -> set[int]:
    # Ignore the initial checkpoint, then treat complete adjacent entries as pre/post decay pairs.
    paired_steps = steps[1:]
    complete_pairs = len(paired_steps) // 2
    return {paired_steps[2 * pair_index] for pair_index in range(complete_pairs)}


PRE_DECAY_CHECKPOINTS = pre_decay_checkpoints(STEPS_A) | pre_decay_checkpoints(STEPS_B)
if FILTER_PRE_DECAY_CHECKPOINTS:
    pre_decay_mask = plot_df["step"].astype(int).isin(PRE_DECAY_CHECKPOINTS)
    plot_df = plot_df[~pre_decay_mask]

task_order = None
if PREFERRED_TASKS:
    available_tasks = set(plot_df["task"])
    task_order = []
    for task in PREFERRED_TASKS:
        if task in available_tasks and task not in task_order:
            task_order.append(task)
    if task_order:
        plot_df["task"] = pd.Categorical(plot_df["task"], categories=task_order, ordered=True)

plot_df = plot_df.sort_values(["task", "model_class", "step"])

if KEEP_LATEST_RUN_PER_MODEL_TASK:
    latest_sort_columns = ["task", "metric", "model_class", "step", "experiment_id"]
    latest_key_columns = ["task", "metric", "model_class", "step"]
    plot_df = (
        plot_df.sort_values(latest_sort_columns, ascending=[True, True, True, True, False])
        .drop_duplicates(latest_key_columns, keep="first")
        .sort_values(["task", "model_class", "step"])
    )

if AGGREGATE_DUPLICATES and not KEEP_LATEST_RUN_PER_MODEL_TASK:
    aggregate_spec = {
        "score": ("score", "mean"),
        "runs": ("experiment_id", "nunique"),
    }
    if COMPUTE_RAW_SCORE_STATS:
        aggregate_spec.update(
            score_n=("score_n", "sum"),
            score_sum=("score_sum", lambda values: values.sum(min_count=1)),
            score_sum_sq=("score_sum_sq", lambda values: values.sum(min_count=1)),
        )

    plot_df = plot_df.groupby(["task", "metric", "model_class", "step"], as_index=False).agg(**aggregate_spec)

    if COMPUTE_RAW_SCORE_STATS:
        variance = (
            plot_df["score_sum_sq"] - (plot_df["score_sum"] ** 2 / plot_df["score_n"])
        ) / (plot_df["score_n"] - 1)
        plot_df["score_std"] = variance.clip(lower=0).pow(0.5)
        plot_df.loc[plot_df["score_n"] <= 1, "score_std"] = pd.NA
        plot_df["score_sem"] = plot_df["score_std"] / plot_df["score_n"].pow(0.5)
        plot_df["score_ci95_normal"] = 1.96 * plot_df["score_sem"]
else:
    plot_df["runs"] = 1

def add_score_error_bounds(plot_df: pd.DataFrame, error_bar: str) -> pd.DataFrame:
    error_columns = {
        "sem": "score_sem",
        "ci95_normal": "score_ci95_normal",
    }
    if not SHOW_SCORE_ERROR_BARS:
        return plot_df
    if error_bar not in error_columns:
        raise ValueError(f"Unknown SCORE_ERROR_BAR={error_bar!r}. Use one of {sorted(error_columns)}.")

    error_column = error_columns[error_bar]
    if error_column not in plot_df.columns:
        return plot_df

    plot_df = plot_df.copy()
    plot_df["score_error"] = plot_df[error_column]
    plot_df["score_error_low"] = (plot_df["score"] - (plot_df["score"] - plot_df["score_error"]).clip(lower=0))
    plot_df["score_error_high"] = ((plot_df["score"] + plot_df["score_error"]).clip(upper=1) - plot_df["score"])
    return plot_df


plot_df = add_score_error_bounds(plot_df, SCORE_ERROR_BAR)


print(f"Plotting {len(plot_df):,} rows across {plot_df['task'].nunique():,} evals")
display(plot_df[plot_df["model_class"] == "transformer 7B"].tail(20))

if plot_df.empty:
    raise ValueError("No rows matched the current filters. Loosen RUN_REGEX, TASK_REGEX, or PREFERRED_TASKS.")

## Accuracy By Training Step

In [ ]:
class MinorAboveThresh(mticker.SymmetricalLogLocator):
    """Standard log-style minor ticks (2-9x per decade), but only
    at or above `thresh` — leaves the linear region tick-free."""
    def __init__(self, base, linthresh, thresh, subs):
        super().__init__(base=base, linthresh=linthresh, subs=subs)
        self.thresh = thresh

    def tick_values(self, vmin, vmax):
        ticks = super().tick_values(vmin, vmax)
        return [t for t in ticks if abs(t) >= self.thresh]

In [ ]:
plot_df["model_class"].unique()

In [ ]:
n_tasks = plot_df["task"].nunique()
col_wrap = 2 if n_tasks <= 4 else 3
facet_cols = min(n_tasks, col_wrap)
facet_height = (
    aes.FIG_HEIGHT_DOUBLE_ROW_IN
    if n_tasks <= col_wrap
    else aes.FIG_HEIGHT_SINGLE_ROW_IN
)
facet_aspect = (aes.PAPER_WIDTH_IN * 1.3 / facet_cols) / facet_height

MODEL_CLASS_PALETTE = aes.set_palette(
    key="models",
    family="OLMo variants",
    color=[
        "#DD0426",
        "#fb4b4e",
        "#FD9B9D",
        "#ce4bfa",
        "#DB97F2",
        "#F1D6FA",
        "#4b88fa",
        "#9cd2fc"
    ],
    names=[
        "transformer 7B",
        "transformer 1B",
        "transformer 275M",
        "hybrid 7B",
        "hybrid 1B",
        "hybrid 275M",
        "GDN± 1B",
        "GDN+ 1B",
    ],
    display_name=lambda _family, name: name,
)

hue_order = [name for name in MODEL_CLASS_PALETTE if name in set(plot_df["model_class"])]
MODEL_CLASS_MARKERS = dict(zip(hue_order, ["o", "o", "o", "s", "s", "s", "D", "D"], strict=False))

g = sns.relplot(
    data=plot_df,
    x="step",
    y="score",
    hue="model_class",
    hue_order=hue_order,
    palette=MODEL_CLASS_PALETTE,
    style="model_class",
    style_order=hue_order,
    col="task",
    col_wrap=col_wrap,
    kind="line",
    marker="o",
    # size=5,
    facet_kws={"sharey": False, "sharex": True},
    col_order=task_order,
    legend=False,
    height=facet_height,
    aspect=facet_aspect,
)
g.set_axis_labels("Training step", "Score")
g.set_titles("{col_name}", fontweight="regular", size=8.0)

g.set(ylim=(0,1))
LINTHRESH = 1000  # tune to your smallest non-zero step spacing

for ax in g.axes.flat:
    ax.set_xscale("symlog", linthresh=LINTHRESH, linscale=0.5)

if SHOW_SCORE_ERROR_BARS and {"score_error_low", "score_error_high"}.issubset(plot_df.columns):
    for task, ax in zip(g.col_names, g.axes.flat, strict=False):
        task_df = plot_df[plot_df["task"].astype(str) == str(task)]
        for model_class, model_df in task_df.groupby("model_class", observed=True):
            model_df = model_df.dropna(subset=["score_error_low", "score_error_high"]).sort_values("step")
            if model_df.empty:
                continue
            ax.errorbar(
                model_df["step"],
                model_df["score"],
                yerr=[model_df["score_error_low"], model_df["score_error_high"]],
                fmt="none",
                color=MODEL_CLASS_PALETTE[model_class],
                alpha=0.35,
                elinewidth=0.8,
                capsize=1.5,
                capthick=0.8,
                zorder=1,
            )


legend_handles = [
    plt.Line2D(
        [0],
        [0],
        color=MODEL_CLASS_PALETTE[model_class],
        marker=MODEL_CLASS_MARKERS.get(model_class, "o"),
        linewidth=1.5,
        markersize=4,
        label=model_class,
    )
    for model_class in hue_order
]

MONOID_CHANCE_BY_BASE_TASK = {
    "monoid_s2": 1 / 2,
    "monoid_a3": 1 / 3,
    # All other current monoid tasks use 4 choices: gold + 3 distractors.
}

COMMUTATIVE_MONOIDS = [
    "monoid_s2",
    "monoid_a3",
    "monoid_z6",
    "monoid_z12",
    "monoid_z24",
    "monoid_z60",
    "monoid_z120",
    "monoid_z360",
    "monoid_z720",
]
SOLUBLE_MONOIDS = ["monoid_s3", "monoid_s4", "monoid_a4"]
INSOLUBLE_MONOIDS = ["monoid_s5", "monoid_s6", "monoid_a5", "monoid_a6"]


def monoid_task_chance(task: str) -> float | None:
    base_task = task.split(":", 1)[0]
    if not base_task.startswith("monoid_"):
        return None
    return MONOID_CHANCE_BY_BASE_TASK.get(base_task, 1 / 4)


def mean_chance(tasks: list[str]) -> float:
    return sum(monoid_task_chance(task) for task in tasks) / len(tasks)


CHANCE_BY_TASK = defaultdict(
    lambda: 0.25,
    {
        "monoid:commutative:3shot": mean_chance(COMMUTATIVE_MONOIDS),
        "monoid:soluble:3shot": mean_chance(SOLUBLE_MONOIDS),
        "monoid:insoluble:3shot": mean_chance(INSOLUBLE_MONOIDS),
    },
)

for task, ax in zip(g.col_names, g.axes.flat, strict=False):
    chance = monoid_task_chance(task)
    if chance is None:
        chance = CHANCE_BY_TASK[task]
    ax.axhline(
        chance,
        color=aes.NONSEMANTIC_COLOR,
        linestyle="--",
        linewidth=1,
        alpha=0.75,
    )

    # Explicit ticks: 0, then clean log decades. Avoids overlap/clutter
    # symlog's default locator introduces near the linear region.
    xmin, xmax = ax.get_xlim()
    decade_ticks = [10**k for k in range(3, int(np.log10(xmax)) + 1)]
    ax.set_xticks([0] + decade_ticks)
    ax.xaxis.set_minor_locator(
        MinorAboveThresh(base=10, linthresh=LINTHRESH, thresh=LINTHRESH, subs=np.arange(2, 10))
    )
    ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)

    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.set_ylim(bottom=0)
    ax.grid(True, alpha=0.3)

for row_start in range(0, len(g.axes.flat), col_wrap):
    row_axes = [ax for ax in g.axes.flat[row_start : row_start + col_wrap] if ax.get_visible()]
    if not row_axes:
        continue
    row_axes[-1].legend(
        handles=legend_handles,
        title="Model class",
        loc="center left",
        bbox_to_anchor=(1.03, 0.5),
        frameon=False,
        fontsize="x-small",
        title_fontsize="x-small",
        borderaxespad=0.0,
        handlelength=1.6,
    )
plt.show()


## Cube Summary Plots

In [ ]:
import matplotlib.gridspec as gridspec  # noqa: E402
import matplotlib.lines as mlines  # noqa: E402

CUBE_TASKS = [
    "formal_langs_cube_unique:v6",
    "formal_langs_cube_reassign_const:v6",
    "formal_langs_cube_reassign_var:v6",
]

TASK_PROPERTIES = {
    "formal_langs_cube_unique:v6": {"name": "Unique", "algebraic": "R-trivial"},
    "formal_langs_cube_reassign_const:v6": {"name": "Constant Reassignment", "algebraic": "Aperiodic"},
    "formal_langs_cube_reassign_var:v6": {"name": "Variable Reassignment", "algebraic": "Periodic"},
}

cube_df = plot_df[plot_df["task"].isin(CUBE_TASKS)].copy()

if isinstance(cube_df["task"].dtype, pd.CategoricalDtype):
    cube_df["task"] = cube_df["task"].cat.remove_unused_categories()

base_task = cube_df["task"].map(lambda t: t)
cube_df["name"] = base_task.map({t: v["name"] for t, v in TASK_PROPERTIES.items()})
cube_df["algebraic"] = base_task.map({t: v["algebraic"] for t, v in TASK_PROPERTIES.items()})

algebraic_order = ["R-trivial", "Aperiodic", "Periodic"]
algebraic_order = [a for a in algebraic_order if a in set(cube_df["algebraic"])]

model_size_order = ["275M", "1B", "7B"]
cube_df["model_size"] = cube_df["model_class"].str.extract(r"(\d+[MB])")[0]
cube_df["model_size"] = pd.Categorical(cube_df["model_size"], categories=model_size_order, ordered=True)
model_size_order = [m for m in model_size_order if m in set(cube_df["model_size"].dropna())]

RATE_7B = 4_194_304
RATE_1B = 1_572_864
RATE_275M = 2_600_000

cube_df["tokens"] = cube_df.apply(
    lambda row: row["step"] * (RATE_7B if row["model_size"] == "7B" else RATE_1B if row["model_size"] == "1B" else RATE_275M),
    axis=1,
)
cube_df = cube_df[cube_df["tokens"] > 0]

n_rows, n_cols = len(model_size_order), len(algebraic_order)

panel_w, panel_h = 1.6, 1.4
legend_w = 1.8
fig_w = panel_w * n_cols + legend_w
fig_h = panel_h * n_rows

fig = plt.figure(figsize=(fig_w, fig_h))
gs = gridspec.GridSpec(
    n_rows, n_cols + 1,
    width_ratios=[panel_w] * n_cols + [legend_w],
    wspace=0.15, hspace=0.35,
    figure=fig,
)

axes = np.empty((n_rows, n_cols), dtype=object)
for r in range(n_rows):
    for c in range(n_cols):
        ax = fig.add_subplot(
            gs[r, c],
            sharex=axes[0, 0] if (r, c) != (0, 0) else None,
            sharey=axes[0, 0] if (r, c) != (0, 0) else None,
        )
        axes[r, c] = ax

for r, model_size in enumerate(model_size_order):
    row_df = cube_df[cube_df["model_size"] == model_size]
    row_present = [mc for mc in hue_order if mc in set(row_df["model_class"])]

    for c, algebraic in enumerate(algebraic_order):
        ax = axes[r, c]
        panel_df = row_df[row_df["algebraic"] == algebraic]
        present = [mc for mc in hue_order if mc in set(panel_df["model_class"])]

        if not panel_df.empty:
            sns.lineplot(
                data=panel_df,
                x="tokens", y="score",
                hue="model_class", hue_order=present,
                style="model_class", style_order=present,
                palette=MODEL_CLASS_PALETTE,
                markers=MODEL_CLASS_MARKERS,
                dashes=False,
                ax=ax,
                legend=False,
                errorbar=("ci", 95),
                err_style="band",
                markersize=4,
                linewidth=1.3,
            )
            for line in ax.get_lines():
                line.set_clip_on(False)

        ax.axhline(0.25, color=aes.NONSEMANTIC_COLOR, linestyle="--", linewidth=1, alpha=0.75)
        ax.set_xscale("log")
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.set_xlabel(None)
        ax.set_ylabel(None)

        if r == 0:
            ax.set_title(algebraic, fontweight="regular", size=9.0)
        if c == 0:
            ax.set_ylabel("Score")
        else:
            ax.tick_params(labelleft=False)
        if r == n_rows - 1:
            if c == 0:
                ax.set_xlabel("Pretraining Tokens")
        else:
            ax.tick_params(labelbottom=False)

    # CHANGED: one legend per row instead of one shared legend for the whole figure
    legend_ax = fig.add_subplot(gs[r, -1])
    legend_ax.axis("off")
    handles = [
        mlines.Line2D(
            [0], [0], color=MODEL_CLASS_PALETTE[mc], marker=MODEL_CLASS_MARKERS.get(mc, "o"),
            markersize=3, linewidth=1.5, label=mc,
        )
        for mc in row_present
    ]
    legend_ax.legend(
        handles=handles, loc="center left", frameon=False,
        fontsize="small", title=None,
    )

plt.show()

aes.save_figure(
    FIGURES_DIR / "cube_overview",
    fig=fig,
    save_png=True,
)

## Cube Tasks By Assignment Count

In [ ]:
def request_file_for_task(result: dict, task_name: str, tasks: dict[str, dict]) -> Path | None:
    task = tasks.get(task_name)
    task_hash = task.get("task_hash") if task else None
    if not task_hash:
        return None

    requests_root = RESULTS_PATH.parent / result["experiment_id"] / "main" / "requests"
    matches = sorted(requests_root.glob(f"*/*_{task_hash[-6:]}-requests.jsonl"))
    return matches[0] if matches else None


def cube_task_parts(task_name: str) -> tuple[str | None, str | None]:
    match = re.fullmatch(r"formal_langs_cube_(.+):(v[246])", task_name)
    if not match:
        return None, None
    task_variant = match.group(1).replace("_", " ")
    return task_variant, match.group(2)


def assignment_group_for(num_assignments: int) -> str:
    if CUBE_ASSIGNMENT_BINS is None:
        return str(num_assignments)
    for low, high in CUBE_ASSIGNMENT_BINS:
        if low <= num_assignments <= high:
            return f"{low}-{high}" if low != high else str(low)
    return str(num_assignments)


def cube_assignment_rows_for(row: pd.Series, result: dict) -> tuple[list[dict], list[dict]]:
    task_name = str(row["task"])
    task_variant, version = cube_task_parts(task_name)
    if task_variant is None:
        return [], []

    tasks = task_by_name(result)
    request_path = request_file_for_task(result, task_name, tasks)
    prediction_path = prediction_file_for_task(result, task_name, tasks)
    if request_path is None or prediction_path is None:
        return [], []

    metric = row["metric"]
    scores_by_assignment = defaultdict(list)
    scores_by_assignment_group = defaultdict(list)
    assignment_values_by_group = defaultdict(set)
    with request_path.open() as request_file, prediction_path.open() as prediction_file:
        for request_line, prediction_line in zip(request_file, prediction_file, strict=False):
            request_doc = json.loads(request_line).get("doc", {})
            num_assignments = request_doc.get("num_assignments")
            if num_assignments is None:
                continue

            prediction = json.loads(prediction_line)
            score = metric_value(prediction.get("instance_metrics", {}), metric)
            if score is None:
                continue

            num_assignments = int(num_assignments)
            assignment_group = assignment_group_for(num_assignments)
            scores_by_assignment[num_assignments].append(float(score))
            scores_by_assignment_group[assignment_group].append(float(score))
            assignment_values_by_group[assignment_group].add(num_assignments)

    rows = []
    exact_rows = []
    for num_assignments, scores in scores_by_assignment.items():
        series = pd.Series(scores, dtype="float64")
        score_n = int(series.count())
        score = float(series.mean())
        score_std = series.std(ddof=1)
        score_sem = score_std / math.sqrt(score_n) if score_n > 1 else None
        exact_rows.append(
            {
                "experiment_id": row["experiment_id"],
                "model_class": row["model_class"],
                "step": row["step"],
                "task": task_name,
                "cube_task": task_variant,
                "version": version,
                "num_variables": int(version.removeprefix("v")),
                "num_assignments": num_assignments,
                "assignment_group": assignment_group_for(num_assignments),
                "score": score,
                "score_std": score_std,
                "score_sem": score_sem,
                "score_ci95_normal": 1.96 * score_sem if score_sem is not None else None,
                "score_n": score_n,
                "assignment_granularity": "exact",
            }
        )

    for assignment_group, scores in scores_by_assignment_group.items():
        series = pd.Series(scores, dtype="float64")
        score_n = int(series.count())
        score = float(series.mean())
        score_std = series.std(ddof=1)
        score_sem = score_std / math.sqrt(score_n) if score_n > 1 else None
        assignment_values = sorted(assignment_values_by_group[assignment_group])
        rows.append(
            {
                "experiment_id": row["experiment_id"],
                "model_class": row["model_class"],
                "step": row["step"],
                "task": task_name,
                "cube_task": task_variant,
                "version": version,
                "num_variables": int(version.removeprefix("v")),
                "assignment_group": assignment_group,
                "assignment_min": assignment_values[0],
                "assignment_max": assignment_values[-1],
                "score": score,
                "score_std": score_std,
                "score_sem": score_sem,
                "score_ci95_normal": 1.96 * score_sem if score_sem is not None else None,
                "score_n": score_n,
                "assignment_granularity": "binned",
            }
        )
    return rows, exact_rows


result_by_id = {result["experiment_id"]: result for result in raw_results}
cube_source_df = plot_df[
    plot_df["task"].astype(str).str.fullmatch(r"formal_langs_cube_.+:v[246]")
].copy()

cube_assignment_rows = []
cube_assignment_exact_rows = []
for _, row in cube_source_df.iterrows():
    binned_rows, exact_rows = cube_assignment_rows_for(row, result_by_id[row["experiment_id"]])
    cube_assignment_rows.extend(binned_rows)
    cube_assignment_exact_rows.extend(exact_rows)

cube_assignment_df = pd.DataFrame(cube_assignment_rows)
cube_assignment_exact_df = pd.DataFrame(cube_assignment_exact_rows)

if cube_assignment_df.empty:
    raise ValueError("No cube assignment metadata matched the current filters.")

cube_assignment_df["step"] = pd.to_numeric(cube_assignment_df["step"], errors="coerce")
cube_assignment_df = cube_assignment_df.sort_values(
    ["version", "cube_task", "assignment_group", "model_class", "step"]
)
cube_assignment_df = add_score_error_bounds(cube_assignment_df, SCORE_ERROR_BAR)

assignment_order = sorted(
    cube_assignment_df["assignment_group"].dropna().unique(),
    key=lambda label: int(str(label).split("-", 1)[0]),
)
version_order = [version for version in ["v2", "v4", "v6"] if version in set(cube_assignment_df["version"])]
cube_task_order = [
    task for task in ["unique", "reassign const", "reassign var", "undefined"]
    if task in set(cube_assignment_df["cube_task"])
]
hue_order = [name for name in MODEL_CLASS_PALETTE if name in set(cube_assignment_df["model_class"])]
assignment_markers = ["o", "X", "s", "D", "P", "^", "v", "<", ">"]
assignment_dashes = ["", (2, 1), (4, 1), (1, 1), (3, 1, 1, 1), (5, 2), (2, 2, 1, 2), (6, 1), (3, 3)]

print(f"Using cube assignment bins: {CUBE_ASSIGNMENT_BINS}")
print(
    f"Plotting {len(cube_assignment_df):,} cube assignment rows across "
    f"{cube_assignment_df['task'].nunique():,} cube tasks and "
    f"{cube_assignment_df['assignment_group'].nunique():,} assignment groups"
)
# display(cube_assignment_df.head(20))

for cube_task in cube_task_order:
    task_df = cube_assignment_df[cube_assignment_df["cube_task"] == cube_task]
    if task_df.empty:
        continue

    g_cube = sns.relplot(
        data=task_df[task_df["step"] < 200_000],
        x="step",
        y="score",
        hue="model_class",
        hue_order=hue_order,
        palette=MODEL_CLASS_PALETTE,
        style="model_class",
        style_order=hue_order,
        col="version",
        col_order=version_order,
        row="assignment_group",
        row_order=assignment_order,
        kind="line",
        marker="o",
        markersize=3,
        linewidth=1.1,
        facet_kws={"sharey": False, "sharex": True},
        legend=False,
        height=aes.FIG_HEIGHT_SINGLE_ROW_IN,
        aspect=2,
    )
    g_cube.set_axis_labels("Training step", "Score")
    g_cube.set_titles(row_template="{row_name} assignments", col_template="{col_name}", fontweight="regular", size=8.0)
    g_cube.set(ylim=(0, 1))

    if SHOW_SCORE_ERROR_BARS and {"score_error_low", "score_error_high"}.issubset(task_df.columns):
        for (assignment_group, version), ax in g_cube.axes_dict.items():
            facet_df = task_df[
                (task_df["assignment_group"] == assignment_group)
                & (task_df["version"] == version)
            ]
            for model_class, model_df in facet_df.groupby("model_class", observed=True):
                model_df = model_df[model_df["step"] < 200_000].dropna(subset=["score_error_low", "score_error_high"]).sort_values("step")
                if model_df.empty:
                    continue
                ax.errorbar(
                    model_df["step"],
                    model_df["score"],
                    yerr=[model_df["score_error_low"], model_df["score_error_high"]],
                    fmt="none",
                    color=MODEL_CLASS_PALETTE[model_class],
                    alpha=0.25,
                    elinewidth=0.7,
                    capsize=1.0,
                    capthick=0.7,
                    zorder=1,
                )

    for ax in g_cube.axes.flat:
        ax.axhline(0.25, color=aes.NONSEMANTIC_COLOR, linestyle="--", linewidth=1, alpha=0.75)
        ax.xaxis.set_major_formatter(aes.COMPACT_NUMBER_FORMATTER)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.3)

    model_handles = [
        plt.Line2D(
            [0],
            [0],
            color=MODEL_CLASS_PALETTE[model_class],
            marker=MODEL_CLASS_MARKERS.get(model_class, "o"),
            linewidth=1.5,
            markersize=4,
            label=model_class,
        )
        for model_class in hue_order
    ]

    g_cube.figure.legend(
        handles=model_handles,
        title="Model class",
        loc="upper center",
        bbox_to_anchor=(0.5, 1.06),
        ncol=max(1, min(4, len(model_handles))),
        frameon=False,
        fontsize="x-small",
        title_fontsize="x-small",
    )
    g_cube.figure.suptitle(f"Cube Task: {cube_task}", y=1.1, fontsize=10)
    plt.show()

    aes.save_figure(
         FIGURES_DIR / f"cube_task_{cube_task.replace(' ', '_')}",
        fig=g_cube.figure,
        save_png=True,
    )


## Cube Final Accuracy Regressions

In [ ]:
def final_cube_regression_df(cube_df: pd.DataFrame) -> pd.DataFrame:
    final_rows = []
    for model_class, model_df in cube_df.groupby("model_class", observed=True):
        final_step = model_df["step"].max()
        final_rows.append(model_df[model_df["step"] == final_step])
    final_df = pd.concat(final_rows, ignore_index=True)
    final_df = final_df.copy()
    final_df["architecture"] = pd.Categorical(final_df["model_class"])
    final_df["cube_task"] = pd.Categorical(
        final_df["cube_task"],
        categories=[task for task in cube_task_order if task in set(final_df["cube_task"])],
        ordered=True,
    )
    final_df["successes"] = final_df["score"] * final_df["score_n"]
    final_df["failures"] = final_df["score_n"] - final_df["successes"]
    return final_df


def prepare_task_regression_df(task_df: pd.DataFrame) -> pd.DataFrame:
    task_df = task_df.copy()
    task_df["num_variables_centered"] = task_df["num_variables"] - task_df["num_variables"].mean()
    task_df["num_assignments_centered"] = task_df["num_assignments"] - task_df["num_assignments"].mean()
    return task_df


def regression_formula_for_task(cube_task: str) -> str:
    if cube_task == "unique":
        return "score ~ C(architecture) + num_variables_centered"
    return (
        "score ~ C(architecture) + "
        "num_variables_centered * num_assignments_centered"
    )


def fit_cube_task_regressions(task_df: pd.DataFrame, cube_task: str) -> tuple[object, object, str, str, str]:
    regression_formula = regression_formula_for_task(cube_task)
    glm_model = smf.glm(
        formula=regression_formula,
        data=task_df,
        family=sm.families.Binomial(),
        freq_weights=task_df["score_n"],
    )
    try:
        glm = glm_model.fit(cov_type="HC3")
        glm_cov_type = "HC3"
    except Exception as exc:
        print(f"[warn] GLM HC3 covariance failed ({type(exc).__name__}: {exc}); refitting with default covariance.")
        glm = glm_model.fit()
        glm_cov_type = "default"

    wls_model = smf.wls(
        formula=regression_formula,
        data=task_df,
        weights=task_df["score_n"],
    )
    try:
        wls = wls_model.fit(cov_type="HC3")
        wls_cov_type = "HC3"
    except Exception as exc:
        print(f"[warn] WLS HC3 covariance failed ({type(exc).__name__}: {exc}); refitting with default covariance.")
        wls = wls_model.fit()
        wls_cov_type = "default"
    return glm, wls, glm_cov_type, wls_cov_type, regression_formula


def coefficient_table_for_task(
    task_name: str, glm, wls, glm_cov_type: str, wls_cov_type: str, regression_formula: str
) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "cube_task": task_name,
            "formula": regression_formula,
            "glm_cov_type": glm_cov_type,
            "glm_logit_coef": glm.params,
            "glm_logit_se": glm.bse,
            "glm_p": glm.pvalues,
            "wls_cov_type": wls_cov_type,
            "wls_accuracy_coef": wls.params,
            "wls_accuracy_se": wls.bse,
            "wls_p": wls.pvalues,
        }
    )

def slugify(value: str) -> str:
    return re.sub(r"[^a-zA-Z0-9]+", "_", value).strip("_").lower()


def write_markdown_table(df: pd.DataFrame, path: Path) -> None:
    text_df = df.copy()
    for col in text_df.columns:
        text_df[col] = text_df[col].map(
            lambda value: f"{value:.6g}" if isinstance(value, float) else str(value)
        )
    columns = list(text_df.columns)
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]
    for _, row in text_df.iterrows():
        lines.append("| " + " | ".join(row[col] for col in columns) + " |")
    path.write_text("\n".join(lines) + "\n")


def save_cube_regression_outputs(
    output_dir: Path,
    coef_table: pd.DataFrame,
    regression_results: dict[str, dict],
    final_df: pd.DataFrame,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    coef_table.to_csv(output_dir / "cube_regression_coefficients.csv", index=False)
    write_markdown_table(coef_table, output_dir / "cube_regression_coefficients.md")
    final_df.to_csv(output_dir / "cube_final_regression_data.csv", index=False)

    metadata = {
        "n_rows": int(len(final_df)),
        "n_examples": int(final_df["score_n"].sum()),
        "tasks": {},
    }
    for cube_task, result in regression_results.items():
        task_slug = slugify(cube_task)
        task_dir = output_dir / task_slug
        task_dir.mkdir(parents=True, exist_ok=True)

        glm_summary = result["glm"].summary().as_text()
        wls_summary = result["wls"].summary().as_text()
        (task_dir / "glm_summary.txt").write_text(glm_summary)
        (task_dir / "wls_summary.txt").write_text(wls_summary)

        task_data = result["data"]
        metadata["tasks"][cube_task] = {
            "formula": result["formula"],
            "glm_cov_type": result["glm_cov_type"],
            "wls_cov_type": result["wls_cov_type"],
            "n_rows": int(len(task_data)),
            "n_examples": int(task_data["score_n"].sum()),
            "glm_summary": str((task_dir / "glm_summary.txt").relative_to(output_dir)),
            "wls_summary": str((task_dir / "wls_summary.txt").relative_to(output_dir)),
        }

    (output_dir / "cube_regression_metadata.json").write_text(json.dumps(metadata, indent=2))
    print(f"Saved cube regression outputs to {output_dir}")



cube_final_df = final_cube_regression_df(cube_assignment_exact_df)
print(
    f"Final cube regression data: {len(cube_final_df):,} rows, "
    f"{cube_final_df['score_n'].sum():,.0f} evaluated examples"
)
display(
    cube_final_df.groupby(["architecture", "cube_task", "num_variables"], observed=True)
    .agg(
        rows=("score", "size"),
        assignment_min=("num_assignments", "min"),
        assignment_max=("num_assignments", "max"),
        mean_accuracy=("score", "mean"),
    )
    .reset_index()
    .head(20)
)

cube_task_regression_results = {}
coef_tables = []
for cube_task in cube_task_order:
    task_df = cube_final_df[cube_final_df["cube_task"] == cube_task]
    if task_df.empty:
        continue
    task_df = prepare_task_regression_df(task_df)
    glm, wls, glm_cov_type, wls_cov_type, regression_formula = fit_cube_task_regressions(task_df, cube_task)
    cube_task_regression_results[cube_task] = {
        "glm": glm,
        "wls": wls,
        "data": task_df,
        "glm_cov_type": glm_cov_type,
        "wls_cov_type": wls_cov_type,
        "formula": regression_formula,
    }
    coef_tables.append(
        coefficient_table_for_task(cube_task, glm, wls, glm_cov_type, wls_cov_type, regression_formula)
    )

    print("=" * 100)
    print(
        f"Cube task: {cube_task} | rows={len(task_df):,} | "
        f"examples={task_df['score_n'].sum():,.0f}"
    )
    print(f"Formula: {regression_formula}")
    print(f"\nBinomial GLM (covariance: {glm_cov_type})")
    print(glm.summary())
    print(f"\nWeighted least squares (covariance: {wls_cov_type})")
    print(wls.summary())

coef_table = pd.concat(coef_tables).rename_axis("term").reset_index()
save_cube_regression_outputs(
    REGRESSION_RESULTS_DIR,
    coef_table,
    cube_task_regression_results,
    cube_final_df,
)
display(coef_table)


## Coverage Checks

In [ ]:
coverage = (
    df.assign(matches_run=lambda x: x["experiment_name"].str.contains(RUN_REGEX or ".*", case=False, regex=True, na=False))
    .drop_duplicates(["experiment_id", "experiment_name", "model", "step", "model_class"])
    .sort_values(["matches_run", "model_class", "step"], ascending=[False, True, True])
)

display(
    coverage[["matches_run", "experiment_name", "model_class", "step", "model", "status"]]
    .reset_index(drop=True)
)

display(
    df.groupby("task", as_index=False)
    .agg(rows=("score", "size"), models=("model_class", "nunique"), steps=("step", "nunique"))
    .sort_values(["rows", "task"], ascending=[False, True])
)